# Module 05 — Neural Network Foundations
## 05-01: The Perceptron & Linear Classifiers

**Objective:** Implement the Rosenblatt Perceptron from scratch, understand its convergence
guarantee on linearly separable data, and demonstrate its XOR limitation.

**Prerequisites:** 01-02 (Linear Algebra), 01-03 (Calculus & Optimisation),
02-01 (Linear Regression), 02-02 (Logistic Regression)


---
## Part 0 — Setup & Prerequisites

This notebook covers the **Perceptron** — the simplest learnable neural unit and the
conceptual ancestor of every modern deep network.  We will:

1. Build the McCulloch-Pitts neuron and use it to implement logic gates.
2. Implement the Rosenblatt Perceptron learning algorithm from scratch with NumPy.
3. Wrap it as a PyTorch `nn.Module` and train it on MNIST binary classification.
4. Compare against `sklearn.linear_model.Perceptron`.
5. Expose the XOR limitation and show why multiple layers are needed.

**Prerequisites:** 01-02 (Linear Algebra), 01-03 (Calculus), 02-02 (Logistic Regression)


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import warnings
import random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms

from sklearn.datasets import make_classification, make_moons
from sklearn.linear_model import Perceptron as SklearnPerceptron
from typing import Any

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility & Device ──────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
BATCH_SIZE       = 256
NUM_EPOCHS       = 20
LEARNING_RATE    = 1e-1      # Gradient-descent lr for TorchPerceptron
PERCEPTRON_LR    = 0.1       # Step size for NumPy Perceptron update rule
PERCEPTRON_ITER  = 150       # Max passes for NumPy Perceptron
BINARY_CLASS_A   = 0         # MNIST digit mapped to label 0
BINARY_CLASS_B   = 1         # MNIST digit mapped to label 1
N_SAMPLES_TOY    = 400       # Samples per 2-D toy dataset
DATA_DIR         = "../data"


### Part 0.1 — Data Overview

We use two data sources throughout the notebook:

- **2-D toy datasets** (linearly separable blobs, two-moons) — ideal for visualising
  decision boundaries and understanding convergence behaviour.
- **MNIST binary** — digits 0 vs 1, flattened to 784-dimensional vectors — a real-world
  benchmark that lets us compare the Perceptron against `sklearn`.


In [ ]:
# ── Generate 2-D Toy Datasets ──────────────────────────────────────────────────
# Linearly separable: two well-separated Gaussian blobs
X_lin, y_lin = make_classification(
    n_samples=N_SAMPLES_TOY,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=2.0,
    random_state=SEED,
)
y_lin = y_lin.astype(int)

# Non-linearly separable: two interleaved moons
X_moon, y_moon = make_moons(n_samples=N_SAMPLES_TOY, noise=0.25, random_state=SEED)
y_moon = y_moon.astype(int)

# ── Load MNIST and Filter to Binary Classes ────────────────────────────────────
mnist_train_full = torchvision.datasets.MNIST(
    root=DATA_DIR, train=True, download=True, transform=transforms.ToTensor(),
)
mnist_test_full = torchvision.datasets.MNIST(
    root=DATA_DIR, train=False, download=True, transform=transforms.ToTensor(),
)


def filter_mnist_to_binary(
    dataset: torchvision.datasets.MNIST,
    class_a: int,
    class_b: int,
) -> tuple[np.ndarray, np.ndarray]:
    '''Extract flattened pixels and binary labels for two MNIST digit classes.

    Args:
        dataset: Full MNIST dataset (torchvision).
        class_a: Digit class mapped to binary label 0.
        class_b: Digit class mapped to binary label 1.

    Returns:
        Tuple of (pixel_array of shape (n, 784), binary_labels of shape (n,)).
    '''
    targets = np.array(dataset.targets)
    mask    = (targets == class_a) | (targets == class_b)
    images  = dataset.data[mask].numpy().astype(np.float32) / 255.0
    labels  = (targets[mask] == class_b).astype(int)
    return images.reshape(-1, 784), labels


X_mnist_trainval, y_mnist_trainval = filter_mnist_to_binary(
    mnist_train_full, BINARY_CLASS_A, BINARY_CLASS_B,
)
X_mnist_test, y_mnist_test = filter_mnist_to_binary(
    mnist_test_full, BINARY_CLASS_A, BINARY_CLASS_B,
)

# Quick split for NumPy Perceptron experiments (90/10 from trainval)
rng_split = np.random.default_rng(SEED)
n_trainval = len(X_mnist_trainval)
val_size   = n_trainval // 10
idx_perm   = rng_split.permutation(n_trainval)
val_idx    = idx_perm[:val_size]
train_idx  = idx_perm[val_size:]
X_mnist_train_np = X_mnist_trainval[train_idx]
y_mnist_train_np = y_mnist_trainval[train_idx]
X_mnist_val_np   = X_mnist_trainval[val_idx]
y_mnist_val_np   = y_mnist_trainval[val_idx]

print(f"2-D linear:   X_lin  shape={X_lin.shape},  classes={np.unique(y_lin)}")
print(f"2-D moons:    X_moon shape={X_moon.shape}, classes={np.unique(y_moon)}")
print(f"MNIST binary: train_np={X_mnist_train_np.shape}, val_np={X_mnist_val_np.shape}, "
      f"test={X_mnist_test.shape}")
class_0_count = int((y_mnist_trainval == 0).sum())
class_1_count = int((y_mnist_trainval == 1).sum())
print(f"Class balance: digit {BINARY_CLASS_A} = {class_0_count}, "
      f"digit {BINARY_CLASS_B} = {class_1_count}")

# ── Visualise Datasets ─────────────────────────────────────────────────────────
fig_eda, axes_eda = plt.subplots(1, 4, figsize=(16, 4))

for ax, X_plot, y_plot, ttl in zip(
    axes_eda[:2],
    [X_lin, X_moon],
    [y_lin, y_moon],
    ["Linearly Separable (blobs)", "Non-Linearly Separable (moons)"],
):
    colors_eda = np.where(y_plot == 0, "#1f77b4", "#ff7f0e")
    ax.scatter(X_plot[:, 0], X_plot[:, 1], c=colors_eda, s=18, alpha=0.7,
               edgecolors="k", lw=0.2)
    ax.set_title(ttl, fontsize=9, fontweight="bold")
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    p0 = mpatches.Patch(color="#1f77b4", label="Class 0")
    p1 = mpatches.Patch(color="#ff7f0e", label="Class 1")
    ax.legend(handles=[p0, p1], fontsize=7)

for ax, cls_lbl, ttl in zip(
    axes_eda[2:],
    [0, 1],
    [f"Digit {BINARY_CLASS_A} (label 0)", f"Digit {BINARY_CLASS_B} (label 1)"],
):
    idxs_cls  = np.where(y_mnist_trainval == cls_lbl)[0][:16]
    grid_imgs = X_mnist_trainval[idxs_cls].reshape(4, 4, 28, 28)
    mosaic    = np.block([[grid_imgs[r, c] for c in range(4)] for r in range(4)])
    ax.imshow(mosaic, cmap="gray")
    ax.set_title(ttl, fontsize=9, fontweight="bold")
    ax.axis("off")

plt.suptitle("Dataset Overview: 2-D Toy Data and MNIST Binary Classes",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Part 1 — The Perceptron from Scratch

### 1.1 The McCulloch-Pitts Neuron (1943)

The **McCulloch-Pitts (MCP)** neuron is the earliest mathematical model of a biological
neuron.  It accepts binary inputs $\mathbf{x} \in \{0, 1\}^n$, computes a weighted sum,
and *fires* if and only if the sum meets or exceeds a threshold $\theta$:

$$z = \sum_{i=1}^{n} w_i x_i, \qquad \hat{y} = \mathbf{1}[z \geq \theta]$$

Weights are **fixed by hand** — the MCP neuron does not learn.  It is useful for implementing
logic gates but cannot adapt to data.

### 1.2 The Rosenblatt Perceptron (1958)

The Perceptron adds a **weight-update rule**.  For each misclassified sample
$(\mathbf{x}_i, y_i)$ with $y_i \in \{-1, +1\}$:

$$\mathbf{w} \leftarrow \mathbf{w} + \eta \, y_i \, \mathbf{x}_i, \qquad
b \leftarrow b + \eta \, y_i$$

**Convergence Theorem:** If the data are linearly separable, the Perceptron converges in at
most $\left(R / \gamma\right)^2$ updates, where $R = \max_i \|\mathbf{x}_i\|_2$
and $\gamma$ is the geometric margin of the separating hyperplane.


In [ ]:
# ── 1.1 McCulloch-Pitts Neuron ─────────────────────────────────────────────────
def mcp_neuron(
    inputs: np.ndarray,
    weights: np.ndarray,
    threshold: float,
) -> int:
    '''Simulate a McCulloch-Pitts binary neuron with fixed weights.

    Args:
        inputs: Binary input vector of shape (n_inputs,).
        weights: Weight vector of shape (n_inputs,).
            Positive values are excitatory, negative are inhibitory.
        threshold: Firing threshold. Neuron fires iff weighted_sum >= threshold.

    Returns:
        1 if the neuron fires, 0 otherwise.
    '''
    weighted_sum = float(np.dot(inputs, weights))
    return int(weighted_sum >= threshold)


# Implement AND, OR, NOT gates as fixed-weight MCP neurons
binary_pairs = [(0, 0), (0, 1), (1, 0), (1, 1)]

w_and   = np.array([1.0, 1.0]);  theta_and = 2.0   # fire iff both inputs = 1
w_or    = np.array([1.0, 1.0]);  theta_or  = 1.0   # fire iff at least one = 1
w_not   = np.array([-1.0]);      theta_not = 0.0   # fire iff input = 0

print("McCulloch-Pitts Logic Gates (weights fixed by hand)")
print("=" * 42)
for x1, x2 in binary_pairs:
    and_out = mcp_neuron(np.array([x1, x2]), w_and, theta_and)
    or_out  = mcp_neuron(np.array([x1, x2]), w_or,  theta_or)
    not_out = mcp_neuron(np.array([x1]),     w_not, theta_not)
    print(f"  x1={x1}, x2={x2}  |  AND={and_out}  OR={or_out}  NOT(x1)={not_out}")

# ── Visualise Step vs Sigmoid Activation ─────────────────────────────────────
z_vals      = np.linspace(-4, 4, 600)
step_out    = (z_vals >= 0).astype(float)
sigmoid_out = 1.0 / (1.0 + np.exp(-z_vals))
relu_out    = np.maximum(0, z_vals)

fig_act, ax_act = plt.subplots(figsize=(8, 4))
ax_act.plot(z_vals, step_out,    lw=2.5, color="#1f77b4", label="Step (MCP / Perceptron)")
ax_act.plot(z_vals, sigmoid_out, lw=2.0, color="#ff7f0e", ls="--", label="Sigmoid (Logistic)")
ax_act.plot(z_vals, relu_out,    lw=2.0, color="#2ca02c", ls=":",  label="ReLU (preview)")
ax_act.axvline(0, color="gray", lw=1, ls=":")
ax_act.set_xlabel("Weighted input z", fontsize=11)
ax_act.set_ylabel("Activation output", fontsize=11)
ax_act.set_title("Activation Functions: Step vs Sigmoid vs ReLU", fontsize=11,
                  fontweight="bold")
ax_act.set_yticks([0, 0.5, 1])
ax_act.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("\nKey insight: the step function is non-differentiable at z=0.")
print("The Perceptron therefore uses a discrete update rule, not gradient descent.")


### 1.3 Perceptron Learning Algorithm

The algorithm performs an **online (stochastic) update** for each misclassified sample:

```
initialise  w = 0,  b = 0
for each epoch until max_iter:
    for each sample (x_i, y_i):
        if sign(w·x_i + b) ≠ y_i:
            w ← w + η * y_i * x_i
            b ← b + η * y_i
    if no mistakes this epoch → CONVERGED
```

**Convergence Theorem:** The algorithm terminates in at most $(R/\gamma)^2$ updates when
the data are linearly separable ($R$ = max sample norm, $\gamma$ = margin width).
With non-separable data it oscillates indefinitely — we stop after `max_iter` epochs.


In [ ]:
# ── 1.3 Perceptron Class (NumPy) ──────────────────────────────────────────────
class Perceptron:
    '''Perceptron binary classifier from scratch using NumPy.

    Implements the Rosenblatt (1958) online update rule.  Guaranteed to
    converge in a finite number of steps when data are linearly separable.

    Attributes:
        learning_rate: Step size applied to each weight update.
        max_iter: Maximum number of passes over the training data.
        weights: Weight vector of shape (n_features,). Set during fit.
        bias: Scalar bias term. Set during fit.
        errors_per_epoch: Misclassification count recorded after each epoch.
        n_updates_total: Cumulative weight-update steps performed.
    '''

    def __init__(self, learning_rate: float = 0.1, max_iter: int = 100) -> None:
        '''Initialise Perceptron with step size and iteration budget.

        Args:
            learning_rate: Scaling factor for each update (default 0.1).
            max_iter: Maximum passes over training data (default 100).
        '''
        self.learning_rate             = learning_rate
        self.max_iter                  = max_iter
        self.weights: np.ndarray | None = None
        self.bias: float               = 0.0
        self.errors_per_epoch: list[int] = []
        self.n_updates_total: int      = 0

    def fit(self, X: np.ndarray, y: np.ndarray) -> "Perceptron":
        '''Train on binary-labelled data using the Perceptron update rule.

        Labels must be in {0, 1}; internally mapped to {-1, +1}.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Binary labels of shape (n_samples,) with values in {0, 1}.

        Returns:
            self (fitted estimator, supports method chaining).
        '''
        n_samples, n_features    = X.shape
        self.weights             = np.zeros(n_features, dtype=float)
        self.bias                = 0.0
        self.errors_per_epoch    = []
        self.n_updates_total     = 0
        y_signed                 = np.where(y == 1, 1, -1)  # {0,1} -> {-1,+1}

        for epoch in range(self.max_iter):
            n_errors = 0
            for xi, yi in zip(X, y_signed):
                activation = float(np.dot(self.weights, xi)) + self.bias
                prediction = 1 if activation >= 0 else -1
                if prediction != yi:
                    update           = self.learning_rate * float(yi)
                    self.weights    += update * xi
                    self.bias       += update
                    n_errors        += 1
                    self.n_updates_total += 1
            self.errors_per_epoch.append(n_errors)
            if n_errors == 0:
                print(f"  Converged at epoch {epoch + 1}  (0 misclassifications)")
                break
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        '''Predict binary class labels for input samples.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Predicted labels of shape (n_samples,) with values in {0, 1}.
        '''
        activations = X @ self.weights + self.bias
        return (activations >= 0).astype(int)

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        '''Compute raw activation values (signed distance from hyperplane).

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Raw activation values of shape (n_samples,).
        '''
        return X @ self.weights + self.bias

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        '''Compute classification accuracy on labelled data.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: True binary labels of shape (n_samples,).

        Returns:
            Accuracy as a float in [0, 1].
        '''
        return float(np.mean(self.predict(X) == y))


print("Perceptron class defined.")
print("  Methods: fit, predict, decision_function, score")


In [ ]:
# ── Utility: Plot 2-D Decision Boundary ──────────────────────────────────────
def plot_decision_boundary(
    clf: Any,
    X: np.ndarray,
    y: np.ndarray,
    ax: Any,
    title: str = "Decision Boundary",
    padding: float = 0.5,
) -> None:
    '''Shade a 2-D decision boundary for any binary classifier.

    Args:
        clf: Binary classifier with a predict(X: np.ndarray) -> np.ndarray method.
        X: 2-D feature matrix of shape (n_samples, 2).
        y: Binary labels of shape (n_samples,).
        ax: Matplotlib Axes object to draw on.
        title: Title for the axes.
        padding: Extra margin beyond the data range for the mesh grid.
    '''
    x_lo = X[:, 0].min() - padding
    x_hi = X[:, 0].max() + padding
    y_lo = X[:, 1].min() - padding
    y_hi = X[:, 1].max() + padding
    xx, yy = np.meshgrid(
        np.linspace(x_lo, x_hi, 300),
        np.linspace(y_lo, y_hi, 300),
    )
    grid_pts = np.column_stack([xx.ravel(), yy.ravel()])
    Z = clf.predict(grid_pts).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.coolwarm)
    ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1.5)
    colors_bd = np.where(y == 0, "#1f77b4", "#ff7f0e")
    ax.scatter(X[:, 0], X[:, 1], c=colors_bd, s=22, edgecolors="k", lw=0.3, zorder=3)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")


print("plot_decision_boundary defined.")


In [ ]:
# ── 1.4 Train on Linearly Separable Data ─────────────────────────────────────
perc_lin = Perceptron(learning_rate=PERCEPTRON_LR, max_iter=PERCEPTRON_ITER)
perc_lin.fit(X_lin, y_lin)

train_acc_lin = perc_lin.score(X_lin, y_lin)
print(f"Train accuracy (linear 2-D):  {train_acc_lin:.4f}")
print(f"Total weight updates:          {perc_lin.n_updates_total}")
print(f"Epochs to convergence:         {len(perc_lin.errors_per_epoch)}")

fig_lin, axes_lin = plt.subplots(1, 2, figsize=(13, 4))

# Convergence curve
axes_lin[0].plot(
    range(1, len(perc_lin.errors_per_epoch) + 1),
    perc_lin.errors_per_epoch,
    marker="o", markersize=4, lw=2, color="#1f77b4",
)
axes_lin[0].set_xlabel("Epoch", fontsize=11)
axes_lin[0].set_ylabel("Misclassifications", fontsize=11)
axes_lin[0].set_title("Perceptron Convergence (Linearly Separable)",
                       fontsize=11, fontweight="bold")
axes_lin[0].axhline(0, color="gray", ls="--", lw=1)

# Decision boundary with weight vector overlay
plot_decision_boundary(perc_lin, X_lin, y_lin, axes_lin[1],
                       title="Final Decision Boundary (w vector shown)")
if perc_lin.weights is not None and np.linalg.norm(perc_lin.weights) > 1e-9:
    w_norm   = perc_lin.weights / np.linalg.norm(perc_lin.weights)
    center   = X_lin.mean(axis=0)
    arrow_end = center + w_norm * 1.0
    axes_lin[1].annotate(
        "",
        xy=arrow_end, xytext=center,
        arrowprops=dict(arrowstyle="->", color="green", lw=2.5),
    )
    axes_lin[1].text(arrow_end[0] + 0.05, arrow_end[1] + 0.05, "w",
                     color="green", fontsize=13, fontweight="bold")

plt.suptitle("Perceptron on Linearly Separable 2-D Data",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nThe weight vector w is the normal to the decision hyperplane.")
print("The Perceptron update rule rotates w towards correctly classified regions.")


### 1.4b Margin Geometry

The **functional margin** of sample $(\mathbf{x}_i, y_i)$ is $\gamma_i = y_i (\mathbf{w}^\top \mathbf{x}_i + b) / \|\mathbf{w}\|_2$. Points far from the hyperplane have large margins; support vectors (nearest points) have the smallest margin.  The Perceptron does not maximise margin — the SVM does (→ 02-05 SVM).


In [ ]:
# ── Margin Geometry on 2-D Linearly Separable Data ───────────────────────────
# The functional margin of a sample (x, y) w.r.t. weights w and bias b is:
#   gamma_i = y_i * (w . x_i + b) / ||w||
# Positive = correctly classified, negative = misclassified.
# The geometric margin is the minimum functional margin over all samples.

w_norm_val = float(np.linalg.norm(perc_lin.weights)) if perc_lin.weights is not None else 1.0
if w_norm_val < 1e-9:
    w_norm_val = 1.0

activations_lin = perc_lin.decision_function(X_lin)        # (n,) raw activations
y_signed_lin    = np.where(y_lin == 1, 1.0, -1.0)
functional_margins = y_signed_lin * activations_lin / w_norm_val  # geometric margin

correctly_classified = functional_margins > 0
n_correct = int(correctly_classified.sum())

print(f"Geometric margin analysis on 2-D linearly separable data:")
print(f"  Total samples:       {len(y_lin)}")
print(f"  Correctly classified:{n_correct} ({n_correct/len(y_lin)*100:.1f}%)")
print(f"  Min functional margin (support): {functional_margins.min():.4f}")
print(f"  Mean functional margin:          {functional_margins.mean():.4f}")

fig_mg, axes_mg = plt.subplots(1, 2, figsize=(13, 4))

# Plot functional margin distribution
axes_mg[0].hist(functional_margins[y_lin == 0], bins=30, alpha=0.6, label="Class 0",
                color="#1f77b4")
axes_mg[0].hist(functional_margins[y_lin == 1], bins=30, alpha=0.6, label="Class 1",
                color="#ff7f0e")
axes_mg[0].axvline(0, color="red", lw=1.5, ls="--", label="Decision boundary")
axes_mg[0].set_xlabel("Functional margin", fontsize=11)
axes_mg[0].set_ylabel("Count", fontsize=11)
axes_mg[0].set_title("Functional Margin Distribution by Class", fontsize=11,
                      fontweight="bold")
axes_mg[0].legend(fontsize=9)

# Scatter plot coloured by margin magnitude
scatter_mg = axes_mg[1].scatter(
    X_lin[:, 0], X_lin[:, 1],
    c=functional_margins, cmap="RdYlGn",
    s=30, edgecolors="k", lw=0.2,
    vmin=-functional_margins.abs_max if hasattr(functional_margins, "abs_max") else -3,
    vmax=3,
)
plt.colorbar(scatter_mg, ax=axes_mg[1], label="Geometric margin")
# Draw decision boundary line
if perc_lin.weights is not None and abs(perc_lin.weights[1]) > 1e-9:
    x_bd = np.linspace(X_lin[:, 0].min() - 0.5, X_lin[:, 0].max() + 0.5, 100)
    y_bd = -(perc_lin.weights[0] * x_bd + perc_lin.bias) / perc_lin.weights[1]
    axes_mg[1].plot(x_bd, y_bd, "k-", lw=2, label="Decision boundary")
axes_mg[1].set_xlabel("Feature 1"); axes_mg[1].set_ylabel("Feature 2")
axes_mg[1].set_title("Samples Coloured by Geometric Margin", fontsize=11,
                      fontweight="bold")
axes_mg[1].legend(fontsize=8)

plt.suptitle("Margin Geometry — How Far Each Sample Is from the Decision Boundary",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── 1.5 Perceptron on Non-Linearly Separable Data (Two Moons) ─────────────────
perc_moon = Perceptron(learning_rate=PERCEPTRON_LR, max_iter=PERCEPTRON_ITER)
perc_moon.fit(X_moon, y_moon)   # Prints nothing — does NOT converge

train_acc_moon = perc_moon.score(X_moon, y_moon)
print(f"Train accuracy (moons 2-D):  {train_acc_moon:.4f}")
print(f"  Reached max_iter={PERCEPTRON_ITER} — algorithm oscillated, never converged.")
print(f"  Total weight updates: {perc_moon.n_updates_total}")

fig_moon, axes_moon = plt.subplots(1, 2, figsize=(13, 4))

axes_moon[0].plot(
    range(1, len(perc_moon.errors_per_epoch) + 1),
    perc_moon.errors_per_epoch,
    lw=2, color="#d62728", label="Non-linear (moons)",
)
axes_moon[0].plot(
    range(1, len(perc_lin.errors_per_epoch) + 1),
    perc_lin.errors_per_epoch,
    lw=2, ls="--", color="#1f77b4", label="Linear (blobs)",
)
axes_moon[0].set_xlabel("Epoch", fontsize=11)
axes_moon[0].set_ylabel("Misclassifications", fontsize=11)
axes_moon[0].set_title("Convergence: Linear vs Non-Linear Data",
                        fontsize=11, fontweight="bold")
axes_moon[0].legend(fontsize=9)

plot_decision_boundary(perc_moon, X_moon, y_moon, axes_moon[1],
                       title="Best Linear Boundary Found (Insufficient!)")

plt.suptitle("Perceptron Limitation on Non-Linearly Separable Data",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("The learned boundary is always a hyperplane — it cannot curve.")


---
## Part 2 — Assembly & Integration

We wrap the Perceptron concept into two reusable components:

1. **`TorchPerceptron`** (`nn.Module`) — differentiable, trained with BCE + Adam.
2. **`BinaryMNIST`** (`Dataset`) — MNIST filtered to two digit classes.

| | NumPy Perceptron | TorchPerceptron |
|---|---|---|
| Update rule | Rosenblatt (discrete) | Gradient of BCE (continuous) |
| Optimiser | Implicit $\eta y_i \mathbf{x}_i$ step | Adam |
| Activation | Step function | Sigmoid (via BCEWithLogitsLoss) |
| Batching | Online (one sample at a time) | Mini-batch |
| Scalability | Poor (inner Python loop) | GPU-accelerated via CUDA |


In [ ]:
# ── 2.1 TorchPerceptron (nn.Module) ──────────────────────────────────────────
class TorchPerceptron(nn.Module):
    '''Single-layer linear perceptron as a PyTorch nn.Module.

    Equivalent to binary logistic regression when trained with
    BCEWithLogitsLoss; the differentiable counterpart of the NumPy Perceptron.

    Attributes:
        linear: nn.Linear layer mapping input_dim to a single logit.
    '''

    def __init__(self, input_dim: int) -> None:
        '''Initialise TorchPerceptron with a single linear projection.

        Args:
            input_dim: Number of input features (e.g. 784 for flat MNIST).
        '''
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Compute the raw logit for each sample in a batch.

        Args:
            x: Input tensor of shape (batch_size, input_dim).

        Returns:
            Logit tensor of shape (batch_size,).
        '''
        return self.linear(x).squeeze(1)


# ── 2.2 BinaryMNIST (Dataset) ────────────────────────────────────────────────
class BinaryMNIST(Dataset):
    '''MNIST subset filtered to two digit classes for binary classification.

    Images are flattened to 784-dim float vectors and normalised to [0, 1].
    Labels are remapped: class_a -> 0, class_b -> 1.

    Attributes:
        images: Float tensor of shape (n_samples, 784).
        labels: Long tensor of shape (n_samples,) with values in {0, 1}.
    '''

    def __init__(
        self,
        root: str,
        class_a: int,
        class_b: int,
        train: bool = True,
    ) -> None:
        '''Initialise BinaryMNIST by filtering and relabelling MNIST.

        Args:
            root: Root directory for MNIST data (downloaded automatically).
            class_a: Digit class mapped to binary label 0.
            class_b: Digit class mapped to binary label 1.
            train: If True use the training split; else use the test split.
        '''
        super().__init__()
        mnist       = torchvision.datasets.MNIST(
            root=root, train=train, download=True,
            transform=transforms.ToTensor(),
        )
        targets_arr = torch.tensor(mnist.targets)
        mask        = (targets_arr == class_a) | (targets_arr == class_b)
        self.images = mnist.data[mask].float() / 255.0   # (n, 28, 28)
        self.images = self.images.view(-1, 784)           # (n, 784)
        self.labels = (targets_arr[mask] == class_b).long()

    def __len__(self) -> int:
        '''Return total number of samples.

        Returns:
            Sample count for this split.
        '''
        return len(self.labels)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        '''Return the (image, label) pair at the given index.

        Args:
            idx: Integer sample index.

        Returns:
            Tuple of (flat image tensor of shape (784,), label scalar tensor).
        '''
        return self.images[idx], self.labels[idx]


n_params = sum(p.numel() for p in TorchPerceptron(784).parameters())
print(f"TorchPerceptron(784) parameter count: {n_params:,}  (784 weights + 1 bias)")


In [ ]:
# ── 2.3 Sanity Check on 2-D Toy Data ─────────────────────────────────────────
X_lin_t = torch.tensor(X_lin, dtype=torch.float32)
y_lin_t = torch.tensor(y_lin, dtype=torch.float32)

torch_perc_2d   = TorchPerceptron(input_dim=2).to(device)
criterion_sanity = nn.BCEWithLogitsLoss()
optimizer_sanity = torch.optim.Adam(torch_perc_2d.parameters(), lr=LEARNING_RATE)

torch_perc_2d.train()
for step in range(100):
    optimizer_sanity.zero_grad()
    logits_s = torch_perc_2d(X_lin_t.to(device))
    loss_s   = criterion_sanity(logits_s, y_lin_t.to(device))
    loss_s.backward()
    optimizer_sanity.step()

torch_perc_2d.eval()
with torch.no_grad():
    preds_s = (torch_perc_2d(X_lin_t.to(device)) >= 0).long().cpu().numpy()

sanity_acc = float(np.mean(preds_s == y_lin))
print(f"Sanity check — TorchPerceptron 2-D after 100 gradient steps:")
print(f"  Accuracy: {sanity_acc:.4f}  (expected > 0.95 for linearly separable data)")
assert sanity_acc > 0.95, f"Sanity check failed: accuracy {sanity_acc:.4f} < 0.95"
print("Sanity check passed.")


---
## Part 3 — Training on MNIST Binary (Digits 0 vs 1)

We train `TorchPerceptron` on MNIST using the official train/test split.
The official training set is further divided 90/10 into `train_set` / `val_set`.

**Expected performance:** Digits 0 and 1 are visually very distinct — a single linear
layer trained with gradient descent should achieve > 99 % test accuracy.


In [ ]:
# ── Standard Training Functions (Binary Classification) ──────────────────────
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Train for one epoch — binary classification with BCEWithLogitsLoss.

    Args:
        model: The neural network model.
        dataloader: Training DataLoader yielding (inputs, labels) batches.
        optimizer: Optimizer instance.
        criterion: Loss function (BCEWithLogitsLoss expected).
        device: Device to move tensors to.

    Returns:
        Tuple of (average_loss, accuracy) over the epoch.
    '''
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0
    for batch_inputs, batch_targets in dataloader:
        batch_inputs  = batch_inputs.to(device)
        batch_targets = batch_targets.float().to(device)  # BCE requires float
        optimizer.zero_grad()
        logits = model(batch_inputs)
        loss   = criterion(logits, batch_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_inputs.size(0)
        predictions   = (logits >= 0).long()
        correct      += (predictions == batch_targets.long()).sum().item()
        total        += batch_targets.size(0)
    return running_loss / total, correct / total


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Evaluate model loss and accuracy on a dataset split.

    Args:
        model: The neural network model.
        dataloader: DataLoader for validation or test split.
        criterion: Loss function (BCEWithLogitsLoss expected).
        device: Device to move tensors to.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    with torch.no_grad():
        for batch_inputs, batch_targets in dataloader:
            batch_inputs  = batch_inputs.to(device)
            batch_targets = batch_targets.float().to(device)
            logits        = model(batch_inputs)
            loss          = criterion(logits, batch_targets)
            running_loss += loss.item() * batch_inputs.size(0)
            predictions   = (logits >= 0).long()
            correct      += (predictions == batch_targets.long()).sum().item()
            total        += batch_targets.size(0)
    return running_loss / total, correct / total


def plot_training_curves(
    train_losses: list[float],
    val_losses: list[float],
    train_accs: list[float] | None = None,
    val_accs: list[float] | None = None,
) -> None:
    '''Plot training and validation loss (and optionally accuracy) curves.

    Args:
        train_losses: Training loss per epoch.
        val_losses: Validation loss per epoch.
        train_accs: Optional training accuracy per epoch.
        val_accs: Optional validation accuracy per epoch.
    '''
    ncols = 2 if (train_accs is not None and val_accs is not None) else 1
    fig_tc, axes_tc = plt.subplots(1, ncols, figsize=(12 if ncols == 2 else 6, 4))
    if ncols == 1:
        axes_tc = [axes_tc]
    axes_tc[0].plot(train_losses, label="Train")
    axes_tc[0].plot(val_losses,   label="Validation")
    axes_tc[0].set_xlabel("Epoch"); axes_tc[0].set_ylabel("Loss")
    axes_tc[0].set_title("Loss Curve"); axes_tc[0].legend()
    if ncols == 2 and train_accs and val_accs:
        axes_tc[1].plot(train_accs, label="Train")
        axes_tc[1].plot(val_accs,   label="Validation")
        axes_tc[1].set_xlabel("Epoch"); axes_tc[1].set_ylabel("Accuracy")
        axes_tc[1].set_title("Accuracy Curve"); axes_tc[1].legend()
    plt.tight_layout()
    plt.show()


print("train_one_epoch, evaluate, plot_training_curves defined.")


In [ ]:
# ── Build DataLoaders ─────────────────────────────────────────────────────────
full_train_set = BinaryMNIST(DATA_DIR, BINARY_CLASS_A, BINARY_CLASS_B, train=True)
test_set       = BinaryMNIST(DATA_DIR, BINARY_CLASS_A, BINARY_CLASS_B, train=False)

generator  = torch.Generator().manual_seed(SEED)
train_size = int(0.9 * len(full_train_set))
val_size   = len(full_train_set) - train_size
train_set, val_set = torch.utils.data.random_split(
    full_train_set, [train_size, val_size], generator=generator,
)

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)

print(f"Train: {train_size} samples ({len(train_loader)} batches)")
print(f"Val:   {val_size} samples   ({len(val_loader)} batches)")
print(f"Test:  {len(test_set)} samples   ({len(test_loader)} batches)")

# ── Train TorchPerceptron on MNIST Binary ─────────────────────────────────────
model_torch = TorchPerceptron(input_dim=784).to(device)
criterion   = nn.BCEWithLogitsLoss()
optimizer   = torch.optim.Adam(model_torch.parameters(), lr=LEARNING_RATE)

train_losses: list[float] = []
val_losses:   list[float] = []
train_accs:   list[float] = []
val_accs:     list[float] = []
best_val_loss    = float("inf")
best_model_state = None

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(
        model_torch, train_loader, optimizer, criterion, device,
    )
    val_loss, val_acc = evaluate(model_torch, val_loader, criterion, device)
    elapsed = time.time() - t0

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_model_state = {k: v.clone() for k, v in model_torch.state_dict().items()}

    print(f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.2%} | Time: {elapsed:.1f}s")

model_torch.load_state_dict(best_model_state)
test_loss, test_acc = evaluate(model_torch, test_loader, criterion, device)
print(f"\nTest accuracy (best model): {test_acc:.4f}")
plot_training_curves(train_losses, val_losses, train_accs, val_accs)


### 3.2 Library Comparison — sklearn Perceptron

We compare `TorchPerceptron` (BCEWithLogitsLoss + Adam) against
`sklearn.linear_model.Perceptron` (classic update rule) on the same
MNIST binary task.  Both are single-layer linear models trained on
identical data — this validates correctness.


In [ ]:
# ── Compare TorchPerceptron vs sklearn Perceptron ─────────────────────────────
sklearn_perc = SklearnPerceptron(
    eta0=PERCEPTRON_LR,
    max_iter=PERCEPTRON_ITER,
    tol=1e-4,
    random_state=SEED,
    n_jobs=-1,
)
t_sk = time.time()
sklearn_perc.fit(X_mnist_train_np, y_mnist_train_np)
t_sk_elapsed = time.time() - t_sk

sk_val_acc  = accuracy_score(y_mnist_val_np, sklearn_perc.predict(X_mnist_val_np))
sk_test_acc = accuracy_score(y_mnist_test,   sklearn_perc.predict(X_mnist_test))

torch_val_acc  = val_accs[-1]    # last epoch value (best loaded above)
torch_test_acc = test_acc

n_params_torch = int(sum(p.numel() for p in model_torch.parameters()))
n_params_sk    = int(sklearn_perc.coef_.size + sklearn_perc.intercept_.size)

rows = [
    {
        "Model":        "TorchPerceptron (BCE + Adam)",
        "Val Acc":      f"{torch_val_acc:.4f}",
        "Test Acc":     f"{torch_test_acc:.4f}",
        "Parameters":   n_params_torch,
        "Update Rule":  "Gradient descent (BCE)",
    },
    {
        "Model":        "sklearn Perceptron",
        "Val Acc":      f"{sk_val_acc:.4f}",
        "Test Acc":     f"{sk_test_acc:.4f}",
        "Parameters":   n_params_sk,
        "Update Rule":  "Rosenblatt (online)",
    },
]
comparison_df = pd.DataFrame(rows)
print("Library Comparison — MNIST Binary (0 vs 1):")
print(comparison_df.to_string(index=False))
print(f"\nsklearn training time: {t_sk_elapsed:.2f}s")
print("Both models achieve very high accuracy — confirming identical linear model class.")


### 3.3 All Linear Classifiers — Same Family, Different Objectives

The Perceptron, Logistic Regression, and Linear SVM are all **linear classifiers** — they all fit a hyperplane $\mathbf{w}^\top \mathbf{x} + b = 0$.  They differ only in what they optimise: hinge loss (Perceptron, SVM) vs log-likelihood (Logistic).


In [ ]:
# ── Extended Comparison: Perceptron vs LogisticRegression vs Linear SVM ───────
# All three are linear classifiers trained on the same MNIST binary data.
# They differ only in loss function and update rule.

lr_model  = LogisticRegression(max_iter=200, random_state=SEED, C=1.0)
svm_model = LinearSVC(max_iter=2000, random_state=SEED, C=1.0)

t_lr = time.time()
lr_model.fit(X_mnist_train_np, y_mnist_train_np)
t_lr_elapsed = time.time() - t_lr

t_svm = time.time()
svm_model.fit(X_mnist_train_np, y_mnist_train_np)
t_svm_elapsed = time.time() - t_svm

lr_test_acc  = accuracy_score(y_mnist_test, lr_model.predict(X_mnist_test))
svm_test_acc = accuracy_score(y_mnist_test, svm_model.predict(X_mnist_test))

# Build comparison table
ext_rows = [
    {
        "Model":         "TorchPerceptron (BCE + Adam)",
        "Test Acc":      f"{test_acc:.4f}",
        "Loss":          "BCE (sigmoid)",
        "Update":        "Gradient descent",
        "Notes":         "Smooth, differentiable",
    },
    {
        "Model":         "sklearn Perceptron",
        "Test Acc":      f"{sk_test_acc:.4f}",
        "Loss":          "Hinge (perceptron)",
        "Update":        "Online (Rosenblatt)",
        "Notes":         "Convergence only if separable",
    },
    {
        "Model":         "Logistic Regression",
        "Test Acc":      f"{lr_test_acc:.4f}",
        "Loss":          "Log-likelihood (BCE)",
        "Update":        "LBFGS / gradient",
        "Notes":         "Probabilistic outputs",
    },
    {
        "Model":         "Linear SVM",
        "Test Acc":      f"{svm_test_acc:.4f}",
        "Loss":          "Hinge (margin-based)",
        "Update":        "Dual / gradient",
        "Notes":         "Maximum margin",
    },
]
ext_df = pd.DataFrame(ext_rows)
print("Extended Comparison — All Linear Models on MNIST Binary (0 vs 1):")
print(ext_df.to_string(index=False))
print(f"\nLogistic Regression training time: {t_lr_elapsed:.2f}s")
print(f"Linear SVM training time:          {t_svm_elapsed:.2f}s")
print("\nAll four models are linear classifiers — they share the same model family.")
print("Differences are only in the loss optimised and the solution's margin properties.")


---
## Part 4 — Evaluation & Analysis

### 4.1 Confusion Matrix & Classification Report


In [ ]:
# ── 4.1 Confusion Matrix ─────────────────────────────────────────────────────
model_torch.eval()
all_preds:  list[int] = []
all_labels: list[int] = []

with torch.no_grad():
    for batch_inputs, batch_targets in test_loader:
        logits = model_torch(batch_inputs.to(device))
        preds  = (logits >= 0).long().cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch_targets.tolist())

all_preds_arr  = np.array(all_preds)
all_labels_arr = np.array(all_labels)

cm = confusion_matrix(all_labels_arr, all_preds_arr)
hdr_pred0 = "Pred 0"; hdr_pred1 = "Pred 1"
print("Confusion Matrix (rows=True label, cols=Predicted label):")
print(f"  {hdr_pred0:>8s}  {hdr_pred1:>8s}")
for i, row_label in enumerate(["True 0", "True 1"]):
    vals = "  ".join(f"{cm[i, j]:8d}" for j in range(2))
    print(f"  {row_label}: {vals}")

label_names = [f"Digit {BINARY_CLASS_A}", f"Digit {BINARY_CLASS_B}"]
print("\nClassification Report:")
print(classification_report(all_labels_arr, all_preds_arr, target_names=label_names))

fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
im_cm = ax_cm.imshow(cm, cmap="Blues")
plt.colorbar(im_cm, ax=ax_cm)
ax_cm.set_xticks([0, 1]); ax_cm.set_xticklabels(label_names)
ax_cm.set_yticks([0, 1]); ax_cm.set_yticklabels(label_names)
ax_cm.set_xlabel("Predicted", fontsize=11)
ax_cm.set_ylabel("True", fontsize=11)
ax_cm.set_title("TorchPerceptron — Test Confusion Matrix",
                fontsize=11, fontweight="bold")
for i in range(2):
    for j in range(2):
        cell_color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax_cm.text(j, i, str(cm[i, j]), ha="center", va="center",
                   fontsize=13, color=cell_color, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── 4.2 Error Analysis — Misclassified Digits ────────────────────────────────
misclassified_idxs = np.where(all_preds_arr != all_labels_arr)[0]
n_errors           = len(misclassified_idxs)
print(f"Misclassifications on test set: {n_errors} / {len(all_labels_arr)}")
print(f"Error rate:  {n_errors / len(all_labels_arr):.4f}")

# Collect misclassified images directly from test_set.images
all_test_imgs   = test_set.images.numpy()    # (n_test, 784)
all_test_labels = test_set.labels.numpy()

if n_errors > 0:
    show_max  = min(16, n_errors)
    show_idxs = misclassified_idxs[:show_max]
    ncols_e   = 8
    nrows_e   = (show_max + ncols_e - 1) // ncols_e
    fig_err, axes_err = plt.subplots(nrows_e, ncols_e, figsize=(16, 2.5 * nrows_e))
    axes_err_flat = np.array(axes_err).reshape(-1)
    for k, sidx in enumerate(show_idxs):
        img_k   = all_test_imgs[sidx].reshape(28, 28)
        true_k  = int(all_test_labels[sidx])
        pred_k  = int(all_preds_arr[sidx])
        axes_err_flat[k].imshow(img_k, cmap="gray")
        axes_err_flat[k].set_title(f"T:{true_k} P:{pred_k}", fontsize=8, color="red")
        axes_err_flat[k].axis("off")
    for k in range(show_max, len(axes_err_flat)):
        axes_err_flat[k].axis("off")
    plt.suptitle("Misclassified Test Samples (T=True, P=Predicted)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Breakdown by true class
    for cls_val, cls_name in zip([0, 1], label_names):
        cls_errors = np.sum(
            (all_labels_arr[misclassified_idxs] == cls_val)
        )
        print(f"  Errors where true = {cls_name}: {cls_errors}")
else:
    print("No misclassifications — perfect test set accuracy!")


### 4.2b Interpretability — Visualising the Learned Weights

A single-layer linear model's weights can be reshaped into the input space (28×28 for MNIST) and interpreted as a **difference template**: pixels where the model has large positive weights correspond to regions that distinguish class 1 from class 0.


In [ ]:
# ── Weight Visualisation — TorchPerceptron's 'Template' ──────────────────────
# The 784 learned weights can be reshaped to 28x28 and interpreted as a
# weighted template: positive weights (bright) correlate with class 1 pixels,
# negative weights (dark) correlate with class 0 pixels.

model_torch.eval()
weight_vec = model_torch.linear.weight.data.squeeze().cpu().numpy()  # (784,)
bias_val   = float(model_torch.linear.bias.data.cpu().item())
weight_img = weight_vec.reshape(28, 28)

fig_wt, axes_wt = plt.subplots(1, 3, figsize=(14, 4))

# Weight map
im_w = axes_wt[0].imshow(weight_img, cmap="RdBu_r",
                           vmin=-np.abs(weight_img).max(),
                           vmax= np.abs(weight_img).max())
plt.colorbar(im_w, ax=axes_wt[0])
axes_wt[0].set_title("Learned Weight Map (28x28)", fontsize=10, fontweight="bold")
axes_wt[0].axis("off")

# Mean digit images for reference
mean_digit_a = X_mnist_trainval[y_mnist_trainval == 0].mean(axis=0).reshape(28, 28)
mean_digit_b = X_mnist_trainval[y_mnist_trainval == 1].mean(axis=0).reshape(28, 28)
axes_wt[1].imshow(mean_digit_a, cmap="gray")
axes_wt[1].set_title(f"Mean Digit {BINARY_CLASS_A} (class 0)", fontsize=10, fontweight="bold")
axes_wt[1].axis("off")
axes_wt[2].imshow(mean_digit_b, cmap="gray")
axes_wt[2].set_title(f"Mean Digit {BINARY_CLASS_B} (class 1)", fontsize=10, fontweight="bold")
axes_wt[2].axis("off")

plt.suptitle("TorchPerceptron: Learned Weights as a Difference Template",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Correlation between weight map and class templates
diff_template = mean_digit_b.ravel() - mean_digit_a.ravel()
correlation   = float(np.corrcoef(weight_vec, diff_template)[0, 1])
print(f"Weight vector shape:  {weight_vec.shape}")
print(f"Bias value:            {bias_val:.4f}")
print(f"Correlation between weight map and (digit1 - digit0) template: {correlation:.4f}")
print("High correlation confirms: the linear model learns a 'difference template'")
print("between the two digit classes.")

# Distribution of weight magnitudes
fig_wd, ax_wd = plt.subplots(figsize=(8, 3))
ax_wd.hist(weight_vec, bins=80, color="#1f77b4", edgecolor="none", alpha=0.7)
ax_wd.axvline(0, color="red", lw=1.5, ls="--", label="zero")
ax_wd.set_xlabel("Weight value", fontsize=11)
ax_wd.set_ylabel("Count", fontsize=11)
ax_wd.set_title("Distribution of Learned Weights (784 values)", fontsize=11,
                fontweight="bold")
ax_wd.legend()
plt.tight_layout()
plt.show()


### 4.3 The XOR Problem — Why the Perceptron Fails

**XOR** is the prototypical problem that is *not* linearly separable.  The four points
$(0,0), (0,1), (1,0), (1,1)$ with labels $0, 1, 1, 0$ cannot be separated by any
hyperplane in 2-D:

$$\text{XOR}(x_1, x_2) = (x_1 \text{ OR } x_2) \text{ AND NOT } (x_1 \text{ AND } x_2)$$

The Perceptron oscillates indefinitely on XOR.  This limitation, formalised by Minsky &
Papert (1969), was resolved only with **multi-layer networks** trained by backpropagation
(→ 05-02 Multilayer Perceptrons, 05-06 Backpropagation).


In [ ]:
# ── 4.3 XOR Demonstration ─────────────────────────────────────────────────────
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=int)

perc_xor = Perceptron(learning_rate=PERCEPTRON_LR, max_iter=200)
perc_xor.fit(X_xor, y_xor)   # Will NOT print "Converged" — oscillates

xor_acc = perc_xor.score(X_xor, y_xor)
print(f"Perceptron on XOR — accuracy: {xor_acc:.4f}")
print(f"  (Best achievable with a single line: 0.75 — any line misclassifies one point)")
last5 = perc_xor.errors_per_epoch[-5:]
print(f"  Errors last 5 epochs: {last5}  (non-zero — oscillating)")

fig_xor, axes_xor = plt.subplots(1, 2, figsize=(12, 4))

# Show XOR points and why no single line can separate them
xor_labels = ["(0,0)=0", "(0,1)=1", "(1,0)=1", "(1,1)=0"]
xor_colors = ["#1f77b4", "#ff7f0e", "#ff7f0e", "#1f77b4"]
for pt, col, lbl in zip(X_xor, xor_colors, xor_labels):
    axes_xor[0].scatter(*pt, c=col, s=350, zorder=5, edgecolors="k", lw=1.5)
    axes_xor[0].annotate(lbl, pt, textcoords="offset points",
                          xytext=(8, 5), fontsize=9)
x_line = np.linspace(-0.3, 1.3, 100)
for slope, intercept, style, label_line in [
    (-1.0, 1.0, "-",  "y=-x+1"),
    ( 1.0, 0.0, "--", "y=x"),
]:
    axes_xor[0].plot(x_line, slope * x_line + intercept,
                      ls=style, lw=1.5, color="gray", label=label_line)
axes_xor[0].set_xlim(-0.3, 1.3); axes_xor[0].set_ylim(-0.4, 1.5)
axes_xor[0].set_title("XOR is NOT Linearly Separable", fontsize=11, fontweight="bold")
axes_xor[0].set_xlabel("x1"); axes_xor[0].set_ylabel("x2")
axes_xor[0].legend(fontsize=8)

# Perceptron errors over epochs on XOR
axes_xor[1].plot(range(1, len(perc_xor.errors_per_epoch) + 1),
                  perc_xor.errors_per_epoch, lw=2, color="#d62728")
axes_xor[1].axhline(0, color="gray", ls="--", lw=1)
axes_xor[1].set_xlabel("Epoch", fontsize=11)
axes_xor[1].set_ylabel("Misclassifications", fontsize=11)
axes_xor[1].set_title("XOR: Perceptron Oscillates (Never Converges)",
                       fontsize=11, fontweight="bold")

plt.suptitle("The XOR Problem — Perceptron's Fundamental Limitation",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nSolution: stack two layers (05-02 MLP).")
print("A 2-layer network with non-linear activations perfectly solves XOR.")


### 4.4 Ablation Study — Effect of Learning Rate

The learning rate $\eta$ in the Perceptron update controls step size.
Too small: slow convergence; too large: potential oscillation.

We sweep $\eta \in \{0.001, 0.01, 0.1, 0.5, 1.0\}$ on the 2-D linearly separable data
and record: convergence epoch, total updates, and final accuracy.


In [ ]:
# ── 4.4 Learning Rate Ablation on 2-D Linearly Separable Data ─────────────────
lr_sweep = [0.001, 0.01, 0.1, 0.5, 1.0]
ablation_rows: list[dict] = []

fig_ab, axes_ab = plt.subplots(1, 2, figsize=(14, 5))
line_colors_ab = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

for lr_val, col_ab in zip(lr_sweep, line_colors_ab):
    perc_ab = Perceptron(learning_rate=lr_val, max_iter=PERCEPTRON_ITER)
    perc_ab.fit(X_lin, y_lin)
    n_epochs_run = len(perc_ab.errors_per_epoch)
    final_acc    = perc_ab.score(X_lin, y_lin)
    n_updates_ab = perc_ab.n_updates_total
    converged    = perc_ab.errors_per_epoch[-1] == 0

    ablation_rows.append({
        "Learning Rate":    lr_val,
        "Converged":        "Yes" if converged else "No",
        "Epochs":           n_epochs_run,
        "Total Updates":    n_updates_ab,
        "Train Accuracy":   round(final_acc, 4),
    })

    axes_ab[0].plot(
        range(1, n_epochs_run + 1),
        perc_ab.errors_per_epoch,
        label=f"lr={lr_val}",
        lw=1.8,
        color=col_ab,
    )

axes_ab[0].set_xlabel("Epoch", fontsize=11)
axes_ab[0].set_ylabel("Misclassifications", fontsize=11)
axes_ab[0].set_title("Convergence Curves by Learning Rate",
                      fontsize=11, fontweight="bold")
axes_ab[0].axhline(0, color="gray", ls="--", lw=1)
axes_ab[0].legend(fontsize=9)

# Bar chart: total updates (proxy for convergence cost)
ablation_df = pd.DataFrame(ablation_rows)
updates_vals = ablation_df["Total Updates"].values
axes_ab[1].bar(
    [str(lr) for lr in lr_sweep],
    updates_vals,
    color=line_colors_ab,
    edgecolor="k",
    lw=0.8,
)
axes_ab[1].set_xlabel("Learning Rate", fontsize=11)
axes_ab[1].set_ylabel("Total Weight Updates", fontsize=11)
axes_ab[1].set_title("Total Updates to Convergence by LR",
                      fontsize=11, fontweight="bold")

plt.suptitle("Ablation Study: Learning Rate Effect on Perceptron Convergence",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(ablation_df.to_string(index=False))
print("\nKey insight: convergence speed is affected by lr; the final boundary is not.")
print("The Perceptron Convergence Theorem guarantees convergence for any lr > 0")
print("on linearly separable data — but the number of steps may differ greatly.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **The Perceptron is a linear classifier.** Its decision surface is always a hyperplane —
   it cannot represent curves or non-linear boundaries.

2. **Convergence is guaranteed — but only for linearly separable data.**  The Perceptron
   Convergence Theorem bounds updates at $(R/\gamma)^2$.  On non-separable data it
   oscillates indefinitely.

3. **XOR is the canonical failure case.** No hyperplane separates the four XOR points.
   This limitation motivated multi-layer networks (→ 05-02) and backpropagation (→ 05-06).

4. **TorchPerceptron ≡ binary logistic regression** when trained with `BCEWithLogitsLoss`.
   The gradient-based formulation generalises to deeper networks; the classic rule does not.

5. **Learning rate affects convergence speed, not the final decision boundary.**  Any
   positive $\eta$ eventually converges on linearly separable data.

### What's Next

→ **05-02 (Multilayer Perceptrons)** stacks hidden layers and non-linear activations to
solve problems like XOR and move far beyond linear decision boundaries.

### Going Further

- Rosenblatt, F. (1958). *The Perceptron: A probabilistic model for information storage.*
  Psychological Review.
- Minsky, M. & Papert, S. (1969). *Perceptrons.* MIT Press. (§11: XOR limitation)
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning.* §4.1 (Discriminant functions)


In [ ]:
# ── Final Assertions & Summary ────────────────────────────────────────────────
assert perc_lin.score(X_lin, y_lin) > 0.99,     "NumPy Perceptron should achieve >99% on linearly separable 2-D data"
assert test_acc > 0.98,     f"TorchPerceptron MNIST binary test accuracy {test_acc:.4f} should be >98%"
assert sk_test_acc > 0.97,     f"sklearn Perceptron MNIST test accuracy {sk_test_acc:.4f} should be >97%"
assert perc_xor.score(X_xor, y_xor) <= 0.75 + 1e-9,     "Perceptron on XOR should not exceed 75% (linearly inseparable)"
assert len(perc_lin.errors_per_epoch) < PERCEPTRON_ITER,     "Perceptron should converge before max_iter on linearly separable 2-D data"

print("All assertions passed.")
print()
print("=" * 62)
print("05-01 — The Perceptron & Linear Classifiers: SUMMARY")
print("=" * 62)
acc_lin_final  = perc_lin.score(X_lin, y_lin)
acc_moon_final = perc_moon.score(X_moon, y_moon)
print(f"  NumPy Perceptron — linear 2-D:         {acc_lin_final:.4f}")
print(f"  NumPy Perceptron — moons 2-D:          {acc_moon_final:.4f} (no convergence)")
print(f"  TorchPerceptron  — MNIST test:         {test_acc:.4f}")
print(f"  sklearn Perceptron — MNIST test:       {sk_test_acc:.4f}")
print(f"  XOR accuracy (linear classifier max):  {perc_xor.score(X_xor, y_xor):.4f}")
print("=" * 62)
print("Next: 05-02 — Multilayer Perceptrons")
